# Data Quality Report (Polars)

Flags implausible delays, stale observations, pre-trip/post-trip observations, and VM vs stop-call delay disagreement before delay metrics are computed using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

quality = load_polars_script("polars_data_quality_report", "data-quality-report.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
LIMIT = 20
MIN_OBSERVATIONS = 50
TIMEZONE = "Europe/Helsinki"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [2]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    limit = LIMIT
    min_observations = MIN_OBSERVATIONS
    timezone = TIMEZONE

rows = quality.load_quality_rows(Args)
summary = quality.build_quality_summary(rows)
line_report = quality.build_quality_by_line(
    rows,
    min_observations=MIN_OBSERVATIONS,
    limit=LIMIT,
)
examples = quality.build_examples(rows, LIMIT)
summary


quality_check,row_count,pct_rows
str,i64,f64
"""analysis_rows""",5585585,100.0
"""is_implausible_delay""",4001,0.07
"""is_stale_observation""",73822,1.32
"""is_pre_trip_observation""",186131,3.33
"""is_post_trip_observation""",101019,1.81
"""has_stop_call_disagreement""",173682,3.11
"""conservative_excluded_default""",310081,5.55
"""conservative_excluded_with_sto…",432344,7.74


In [3]:
line_report


line_ref,row_count,line_name,implausible_delay_rows,stale_rows,pre_trip_rows,post_trip_rows,stop_call_disagreement_rows,conservative_excluded_rows,conservative_excluded_pct
str,u32,str,u32,u32,u32,u32,u32,u32,f64
"""79A""",3512,"""79A""",0,1168,700,1544,230,2244,63.9
"""N6""",8135,"""N6""",1040,0,1922,2884,3246,4809,59.11
"""711""",6875,"""711""",0,1292,2550,0,16,3842,55.88
"""P3""",9659,"""P3""",0,0,5276,16,30,5292,54.79
"""V2""",2818,"""V2""",0,0,1499,0,9,1499,53.19
…,…,…,…,…,…,…,…,…,…
"""203""",4307,"""203""",0,1003,131,1050,121,1181,27.42
"""L5""",6374,"""L5""",0,0,402,1315,239,1717,26.94
"""V1""",9810,"""V1""",73,0,2383,254,447,2637,26.88


In [4]:
examples


recorded_at_utc,line_ref,direction_ref,vehicle_id,trip_match_key,next_stop_point_ref,delay_min,observation_age_seconds,stop_call_delay_diff_seconds,is_implausible_delay,is_stale_observation,is_pre_trip_observation,is_post_trip_observation,has_stop_call_disagreement
"datetime[μs, UTC]",str,str,str,str,str,f64,i64,f64,bool,bool,bool,bool,bool
2026-04-23 10:50:27 UTC,"""6A""","""1""","""19915""","""6A|1|1776927660""","""1901""",190.1,3,11406.0,true,false,false,true,true
2026-04-23 10:50:58 UTC,"""6A""","""1""","""19915""","""6A|1|1776927660""","""1901""",190.6,2,11436.0,true,false,false,true,true
2026-04-23 10:51:25 UTC,"""6A""","""1""","""19915""","""6A|1|1776927660""","""1901""",191.12,5,11467.0,true,false,false,true,true
2026-04-23 10:51:58 UTC,"""6A""","""1""","""19915""","""6A|1|1776927660""","""1901""",191.62,2,11497.0,true,false,false,true,true
2026-04-23 10:52:28 UTC,"""6A""","""1""","""19915""","""6A|1|1776927660""","""1901""",192.12,2,11527.0,true,false,false,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-04-24 05:24:31 UTC,"""N11""","""1""","""60622""","""N11|1|1777023300""","""3151""",-261.35,4,15681.0,true,false,true,false,true
2026-04-24 05:25:01 UTC,"""N11""","""1""","""60622""","""N11|1|1777023300""","""3151""",-260.75,4,15645.0,true,false,true,false,true
2026-04-24 05:25:31 UTC,"""N11""","""1""","""60622""","""N11|1|1777023300""","""3151""",-260.35,4,15621.0,true,false,true,false,true


In [5]:
chart_data = summary.filter(pl.col("quality_check") != "analysis_rows")
if chart_data.is_empty():
    print("No quality flags found.")
else:
    fig = px.bar(
        chart_data,
        x="row_count",
        y="quality_check",
        orientation="h",
        title="Quality flag row counts",
        labels={"quality_check": "Quality check", "row_count": "Rows"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
